# Step 4: Validation with PUMAS on Real GWAS Data

This notebook implements Step 4 of the research plan: validation of the summary-statistics neural network using PUMAS pseudo-subset splitting on GWAS summary statistics.

**Methods compared:**
1. **C+T** — Clumping + Thresholding (standard baseline)
2. **LDpred2-inf** — Infinitesimal ridge shrinkage
3. **PRS-CS** — Continuous shrinkage prior
4. **Gaussian NN** — Summary-stat neural network under Gaussian assumption (Part 1)
5. **Edgeworth NN** — Summary-stat NN with cumulant corrections (Part 2, ablation)

**Evaluation:** PUMAS pseudo-subset train/validation splitting, with summary-stat R² on validation folds.

We first validate the pipeline on synthetic data (where ground truth is known), then demonstrate the framework for real GWAS traits.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

from ssnn.pumas import generate_pumas_splits, pumas_summary_r2
from ssnn.baselines import clump_and_threshold, ldpred2_inf, prs_cs
from ssnn.pumas_validation import (
    TraitConfig,
    run_validation,
    run_synthetic_validation,
)
from ssnn.cumulants import snp_cumulants

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})

METHOD_COLORS = {
    "C+T": "#8C564B",
    "LDpred2-inf": "#9467BD",
    "PRS-CS": "#E377C2",
    "Gaussian NN": "#DD8452",
    "Edgeworth NN": "#55A868",
}
METHOD_ORDER = ["C+T", "LDpred2-inf", "PRS-CS", "Gaussian NN", "Edgeworth NN"]

## 1. Pipeline Validation on Synthetic Data

Before applying to real GWAS data, we verify the PUMAS validation pipeline produces sensible rankings on synthetic data where the ground truth is known.

In [ ]:
# Run on synthetic data with mixed MAF spectrum
synth_result = run_synthetic_validation(
    p=30, N=20000,
    maf_spectrum="mixed", ld_decay=0.5,
    heritability=0.5, sparsity=0.3,
    n_splits=5, seed=42, verbose=True,
)

print(f"\n--- Synthetic Validation: {synth_result.trait_name} ---")
print(f"{'Method':<16s} {'Mean R²':>8s} {'Std R²':>8s} {'Best Params'}")
print("-" * 60)
for method in METHOD_ORDER:
    if method in synth_result.best_per_method:
        info = synth_result.best_per_method[method]
        print(f"{method:<16s} {info['mean_r2']:8.4f} {info['std_r2']:8.4f} {info['params']}")

In [ ]:
# Bar chart of best validation R² per method
methods = [m for m in METHOD_ORDER if m in synth_result.best_per_method]
means = [synth_result.best_per_method[m]["mean_r2"] for m in methods]
stds = [synth_result.best_per_method[m]["std_r2"] for m in methods]
colors = [METHOD_COLORS[m] for m in methods]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(methods, means, yerr=stds, capsize=5,
              color=colors, edgecolor="black", linewidth=0.5)
ax.set_ylabel("PUMAS Validation $R^2$")
ax.set_title(f"Synthetic GWAS: {synth_result.trait_name}")
for bar, mean in zip(bars, means):
    ypos = max(bar.get_height(), 0) + 0.005
    ax.text(bar.get_x() + bar.get_width()/2, ypos,
            f"{mean:.3f}", ha="center", va="bottom", fontsize=10)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 2. Sensitivity to Allele Frequency Spectrum

The Edgeworth correction should matter most when genotypes are skewed (rare variants, MAF far from 0.5). We run the PUMAS pipeline across MAF spectra to verify.

In [ ]:
maf_spectra = ["common", "mixed", "rare"]
maf_results = {}

for spec in maf_spectra:
    print(f"\n=== MAF spectrum: {spec} ===")
    maf_results[spec] = run_synthetic_validation(
        p=30, N=20000,
        maf_spectrum=spec, ld_decay=0.5,
        heritability=0.5, sparsity=0.3,
        n_splits=5, seed=200, verbose=True,
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: R² by method across MAF spectra
ax = axes[0]
x = np.arange(len(maf_spectra))
width = 0.15
for i, method in enumerate(METHOD_ORDER):
    vals = []
    errs = []
    for spec in maf_spectra:
        info = maf_results[spec].best_per_method.get(method, {"mean_r2": 0, "std_r2": 0})
        vals.append(info["mean_r2"])
        errs.append(info["std_r2"])
    ax.bar(x + i * width, vals, width, yerr=errs, label=method,
           color=METHOD_COLORS[method], edgecolor="black", linewidth=0.3, capsize=3)
ax.set_xticks(x + 2 * width)
ax.set_xticklabels(maf_spectra)
ax.set_xlabel("MAF Spectrum")
ax.set_ylabel("PUMAS Validation $R^2$")
ax.set_title("Method Comparison Across MAF Spectra")
ax.legend(fontsize=8, loc="upper right")

# Panel 2: Edgeworth - Gaussian gap
ax = axes[1]
gaps = []
for spec in maf_spectra:
    gauss = maf_results[spec].best_per_method.get("Gaussian NN", {"mean_r2": 0})["mean_r2"]
    ew = maf_results[spec].best_per_method.get("Edgeworth NN", {"mean_r2": 0})["mean_r2"]
    gaps.append(ew - gauss)
ax.bar(maf_spectra, gaps, color="#55A868", edgecolor="black", linewidth=0.5)
ax.set_xlabel("MAF Spectrum")
ax.set_ylabel("$\\Delta R^2$ (Edgeworth $-$ Gaussian)")
ax.set_title("Edgeworth Correction Benefit (PUMAS)")
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
for i, (spec, gap) in enumerate(zip(maf_spectra, gaps)):
    ax.text(i, gap + 0.001 * np.sign(gap), f"{gap:+.4f}",
            ha="center", va="bottom" if gap >= 0 else "top", fontsize=9)

plt.tight_layout()
plt.show()

## 3. Sensitivity to Sample Size

Larger GWAS produce more precise summary statistics. Does the Edgeworth correction benefit more or less at different sample sizes?

In [ ]:
sample_sizes = [5000, 10000, 20000, 50000]
n_results = {}

for N in sample_sizes:
    print(f"\n=== N = {N} ===")
    n_results[N] = run_synthetic_validation(
        p=30, N=N,
        maf_spectrum="mixed", ld_decay=0.5,
        heritability=0.5, sparsity=0.3,
        n_splits=5, seed=300, verbose=True,
    )

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for method in METHOD_ORDER:
    vals = []
    for N in sample_sizes:
        info = n_results[N].best_per_method.get(method, {"mean_r2": 0})
        vals.append(info["mean_r2"])
    ax.plot(sample_sizes, vals, marker="o", label=method,
            color=METHOD_COLORS[method], linewidth=2)
ax.set_xlabel("GWAS Sample Size (N)")
ax.set_ylabel("PUMAS Validation $R^2$")
ax.set_title("Method Performance vs. GWAS Sample Size")
ax.legend(fontsize=9)
ax.set_xscale("log")
plt.tight_layout()
plt.show()

## 4. Simulated "Real" Traits

We simulate three trait-like scenarios that mimic properties of well-studied traits:
- **Height-like**: high heritability, highly polygenic, common variants
- **BMI-like**: moderate heritability, moderately sparse, mixed MAF
- **LDL-like**: moderate heritability, sparse architecture, includes rare variants

In [ ]:
trait_configs = [
    {"name": "Height-like", "h2": 0.6, "sparsity": 0.5, "maf": "common", "N": 50000},
    {"name": "BMI-like", "h2": 0.3, "sparsity": 0.2, "maf": "mixed", "N": 30000},
    {"name": "LDL-like", "h2": 0.3, "sparsity": 0.1, "maf": "rare", "N": 40000},
]

trait_results = {}
for tc in trait_configs:
    print(f"\n{'='*60}")
    print(f"  Trait: {tc['name']}")
    print(f"{'='*60}")
    trait_results[tc["name"]] = run_synthetic_validation(
        p=50, N=tc["N"],
        maf_spectrum=tc["maf"], ld_decay=0.5,
        heritability=tc["h2"], sparsity=tc["sparsity"],
        n_splits=5, seed=500, verbose=True,
    )

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)

for ax, tc in zip(axes, trait_configs):
    name = tc["name"]
    result = trait_results[name]
    methods = [m for m in METHOD_ORDER if m in result.best_per_method]
    means = [result.best_per_method[m]["mean_r2"] for m in methods]
    stds = [result.best_per_method[m]["std_r2"] for m in methods]
    colors = [METHOD_COLORS[m] for m in methods]

    bars = ax.bar(range(len(methods)), means, yerr=stds, capsize=4,
                  color=colors, edgecolor="black", linewidth=0.5)
    ax.set_xticks(range(len(methods)))
    ax.set_xticklabels(methods, rotation=30, ha="right", fontsize=9)
    ax.set_ylabel("PUMAS Validation $R^2$")
    ax.set_title(f"{name}\n(h²={tc['h2']}, MAF={tc['maf']}, N={tc['N']:,})")
    for bar, mean in zip(bars, means):
        ypos = max(bar.get_height(), 0) + 0.002
        ax.text(bar.get_x() + bar.get_width()/2, ypos,
                f"{mean:.3f}", ha="center", va="bottom", fontsize=8)

plt.suptitle("PUMAS Validation Across Simulated Traits", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## 5. Edgeworth Ablation: The Key Question

The central question from the research plan: **Does the Edgeworth correction for genotype non-Gaussianity improve PRS prediction in the PUMAS validation framework?**

We plot the Edgeworth improvement (ΔR² = Edgeworth − Gaussian) across all scenarios, correlating it with the degree of genotype non-Gaussianity.

In [ ]:
# Gather delta R^2 from all experiments
ablation_data = []

# From MAF sweep
for spec in maf_spectra:
    r = maf_results[spec]
    gauss = r.best_per_method.get("Gaussian NN", {"mean_r2": 0})["mean_r2"]
    ew = r.best_per_method.get("Edgeworth NN", {"mean_r2": 0})["mean_r2"]
    ablation_data.append({
        "scenario": f"MAF={spec}",
        "delta_r2": ew - gauss,
        "category": "MAF sweep",
    })

# From trait simulations
for tc in trait_configs:
    r = trait_results[tc["name"]]
    gauss = r.best_per_method.get("Gaussian NN", {"mean_r2": 0})["mean_r2"]
    ew = r.best_per_method.get("Edgeworth NN", {"mean_r2": 0})["mean_r2"]
    ablation_data.append({
        "scenario": tc["name"],
        "delta_r2": ew - gauss,
        "category": "Trait sim",
    })

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
labels = [d["scenario"] for d in ablation_data]
deltas = [d["delta_r2"] for d in ablation_data]
colors_bar = ["#4C72B0" if d["category"] == "MAF sweep" else "#C44E52" for d in ablation_data]

bars = ax.bar(range(len(labels)), deltas, color=colors_bar,
              edgecolor="black", linewidth=0.5)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=25, ha="right")
ax.set_ylabel("$\\Delta R^2$ (Edgeworth $-$ Gaussian NN)")
ax.set_title("Edgeworth Ablation: Does the Cumulant Correction Help?")
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)

for bar, delta in zip(bars, deltas):
    ypos = delta + 0.001 * np.sign(delta)
    ax.text(bar.get_x() + bar.get_width()/2, ypos,
            f"{delta:+.4f}", ha="center",
            va="bottom" if delta >= 0 else "top", fontsize=9)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor="#4C72B0", edgecolor="black", label="MAF sweep"),
    Patch(facecolor="#C44E52", edgecolor="black", label="Trait simulation"),
], fontsize=9)

plt.tight_layout()
plt.show()

## 6. Summary Table

Comprehensive comparison across all methods and scenarios.

In [ ]:
all_experiments = [
    ("Synth default", synth_result),
] + [
    (f"MAF={spec}", maf_results[spec]) for spec in maf_spectra
] + [
    (f"N={N}", n_results[N]) for N in sample_sizes
] + [
    (tc["name"], trait_results[tc["name"]]) for tc in trait_configs
]

header = f"{'Scenario':<22s}"
for method in METHOD_ORDER:
    header += f" {method:>14s}"
header += f" {'ΔR² (EW-G)':>12s}"
print(header)
print("-" * len(header))

for name, result in all_experiments:
    row = f"{name:<22s}"
    gauss_r2 = 0.0
    ew_r2 = 0.0
    for method in METHOD_ORDER:
        info = result.best_per_method.get(method, {})
        r2 = info.get("mean_r2", float("nan"))
        row += f" {r2:14.4f}"
        if method == "Gaussian NN":
            gauss_r2 = r2
        if method == "Edgeworth NN":
            ew_r2 = r2
    delta = ew_r2 - gauss_r2
    row += f" {delta:+12.4f}"
    print(row)

print()
print("ΔR² = Edgeworth NN R² − Gaussian NN R² (positive = Edgeworth correction helps)")
print("All R² values are PUMAS validation R² (summary-stat based, best hyperparameters)")

## 7. Template for Real GWAS Data

To apply this pipeline to real GWAS summary statistics, provide:
- `Sigma_beta`: marginal association vector (p,) — typically from `X^T y / N`
- `Sigma`: LD matrix (p, p) — from a reference panel (e.g., 1000 Genomes)
- `maf`: minor allele frequencies (p,) — from the GWAS or reference panel
- `E_y2`: phenotypic variance (scalar) — use 1.0 for standardized traits
- `N`: GWAS sample size

In [ ]:
# --- Example: Loading real GWAS data (uncomment and adapt paths) ---
#
# import numpy as np
# from ssnn.pumas_validation import TraitConfig, run_validation
#
# # Load your data
# Sigma = np.load("path/to/ld_matrix.npy")
# Sigma_beta = np.load("path/to/marginal_associations.npy")
# maf = np.load("path/to/maf.npy")
# N = 350000  # e.g., UK Biobank
# E_y2 = 1.0  # standardized phenotype
#
# config = TraitConfig(
#     name="Height (UKB)",
#     Sigma_beta=Sigma_beta,
#     Sigma=Sigma,
#     maf=maf,
#     E_y2=E_y2,
#     N=N,
# )
#
# result = run_validation(config, n_splits=5, verbose=True)
#
# for method, info in result.best_per_method.items():
#     print(f"{method}: R² = {info['mean_r2']:.4f} ± {info['std_r2']:.4f}")